# Модель SIR: Многокритериальная оптимизация параметров

**Цель работы:** Найти оптимальные комбинации параметров, минимизирующие
одновременно два критерия: пиковую заболеваемость и долю умерших.

Используется эволюционный алгоритм Borg MOEA из пакета BlackBoxOptim.

## 1. Инициализация проекта и загрузка пакетов

In [ ]:
using DrWatson
@quickactivate "project"
using BlackBoxOptim, Random, Statistics
include(srcdir("sir_model.jl"))

## 2. Целевая функция

Функция `cost_multi(x)` принимает вектор параметров:
- `x[1]` — β_und (коэффициент заражения для невыявленных)
- `x[2]` — detection_time (время до выявления заболевания)
- `x[3]` — death_rate (вероятность летального исхода)

Возвращает кортеж из двух минимизируемых критериев:
- средняя пиковая заболеваемость (доля инфицированных в максимуме)
- средняя доля умерших

Для надёжности выполняется 3 повторных прогона с разными seed.

In [ ]:
function cost_multi(x)
    replicates = 3
    peak_vals = Float64[]
    dead_vals = Float64[]

    for rep in 1:replicates
        model = initialize_sir(;
            Ns = [1000, 1000, 1000],
            β_und = fill(x[1], 3),
            β_det = fill(x[1]/10, 3),
            infection_period = 14,
            detection_time = round(Int, x[2]),
            death_rate = x[3],
            reinfection_probability = 0.1,
            Is = [0, 0, 1],
            seed = 42 + rep,
        )

        infected_frac(model) = count(a.status == :I for a in allagents(model)) / nagents(model)
        peak = 0.0

        for step in 1:100
            Agents.step!(model, 1)
            frac = infected_frac(model)
            if frac > peak
                peak = frac
            end
        end

        deaths = 3000 - nagents(model)
        push!(peak_vals, peak)
        push!(dead_vals, deaths / 3000)
    end

    return (mean(peak_vals), mean(dead_vals))
end

## 3. Запуск оптимизации

Параметры оптимизации:
- `Method = :borg_moea` — алгоритм Borg MOEA для многокритериальной оптимизации
- `FitnessScheme` — схема минимизации обоих критериев
- `SearchRange` — границы поиска для каждого параметра:
  - β_und: от 0.1 до 1.0
  - detection_time: от 3 до 14 дней
  - death_rate: от 0.01 до 0.1 (1%–10%)
- `NumDimensions = 3` — три оптимизируемых параметра
- `MaxTime = 60` — максимальное время оптимизации (60 секунд)

In [ ]:
result = bboptimize(
    cost_multi,
    Method = :borg_moea,
    FitnessScheme = ParetoFitnessScheme{2}(is_minimizing=true),
    SearchRange = [(0.1, 1.0), (3.0, 14.0), (0.01, 0.1)],
    NumDimensions = 3,
    MaxTime = 60,
    TraceMode = :compact,
)

## 4. Извлечение результатов

- `best_candidate(result)` — оптимальные параметры
- `best_fitness(result)` — соответствующие значения критериев

In [ ]:
best = best_candidate(result)
fitness = best_fitness(result)

## 5. Вывод оптимальных параметров

In [ ]:
println("\nОптимальные параметры:")
println("β_und = $(round(best[1], digits=3))")
println("Время выявления = $(round(Int, best[2])) дней")
println("Смертность = $(round(best[3], digits=4))")
println("\nПик заболеваемости: $(round(fitness[1], digits=4))")
println("Доля умерших: $(round(fitness[2], digits=4))")